In [2]:
from pathlib import Path
import csv, re, warnings
import numpy as np
import pandas as pd
from IPython.display import display

ALPHA = 0.05
CANDIDATES = [Path('/mnt/data/data.csv'), Path('data.csv'), Path('../data.csv')]
csv_path = next((p for p in CANDIDATES if p.exists()), None)
if csv_path is None:
    try:
        from ipywidgets import FileUpload
        from IPython.display import display as _display
        print('No standard data.csv path found. Set CSV_PATH below to your file path, or upload data.csv using your notebook interface.')
        CSV_PATH = None  # Example: CSV_PATH = r'C:/path/to/data.csv'
        if CSV_PATH:
            csv_path = Path(CSV_PATH)
    except Exception:
        CSV_PATH = None
        print('No standard data.csv path found. Set CSV_PATH below to your file path.')

if csv_path is None or not csv_path.exists():
    raise FileNotFoundError('Place data.csv in /mnt/data or the notebook folder, or set CSV_PATH to its location.')

# Sniff the delimiter, with common dataset delimiters as fallback.
sample = csv_path.read_text(encoding='utf-8-sig', errors='replace')[:8192]
try:
    delimiter = csv.Sniffer().sniff(sample, delimiters=',;\t|').delimiter
except csv.Error:
    delimiter = ';' if sample.count(';') > sample.count(',') else ','
df = pd.read_csv(csv_path, sep=delimiter, encoding='utf-8-sig', na_values=['?', 'NA', 'N/A', ''])
df.columns = [str(c).strip().replace('\ufeff','') for c in df.columns]
print(f'File: {csv_path.resolve()} | delimiter: {delimiter!r} | shape: {df.shape}')
display(df.head())
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'distinct': df.nunique(dropna=True)}))

File: D:\Data_Analysis\data.csv | delimiter: ';' | shape: (4424, 37)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


,dtype,missing,distinct
Marital status,int64,0,6
Application mode,int64,0,18
Application order,int64,0,8
Course,int64,0,17
Daytime/evening attendance,int64,0,2
Previous qualification,int64,0,17
Previous qualification (grade),float64,0,101
Nacionality,int64,0,21
Mother's qualification,int64,0,29
Father's qualification,int64,0,34


In [3]:
def key(name):
    return re.sub(r'[^a-z0-9]+', '', str(name).lower())

column_lookup = {key(c): c for c in df.columns}
def find_column(*aliases, required=True):
    for alias in aliases:
        hit = column_lookup.get(key(alias))
        if hit is not None: return hit
    if required: raise KeyError(f'Could not find any of these columns: {aliases}. Available columns: {list(df.columns)}')
    return None

COL = {
 'target': find_column('Target'),
 'tuition': find_column('Tuition fees up to date', 'Tuition fees up to date (1=yes; 0=no)'),
 'scholarship': find_column('Scholarship holder', 'Scholarship holder (1=yes; 0=no)'),
 'age': find_column('Age at enrollment'),
 'admission_grade': find_column('Admission grade'),
 'previous_grade': find_column('Previous qualification (grade)'),
}
print('Matched columns:', COL)
print(f'Rows: {len(df):,} | columns: {df.shape[1]} | missing cells: {int(df.isna().sum().sum()):,} | exact duplicate rows: {int(df.duplicated().sum()):,}')

# Clean target labels and ensure 0/1 flags are categorical values where possible.
data = df.copy()
data[COL['target']] = data[COL['target']].astype('string').str.strip()
for name in ['tuition','scholarship']:
    # Convert common string forms to numeric; keep unknown values missing.
    data[COL[name]] = pd.to_numeric(data[COL[name]], errors='coerce')
for name in ['age','admission_grade','previous_grade']:
    data[COL[name]] = pd.to_numeric(data[COL[name]], errors='coerce')
display(data[[COL[k] for k in COL]].isna().sum().rename('missing values').to_frame())
display(data[COL['target']].value_counts(dropna=False).rename_axis('Target').to_frame('count'))

Matched columns: {'target': 'Target', 'tuition': 'Tuition fees up to date', 'scholarship': 'Scholarship holder', 'age': 'Age at enrollment', 'admission_grade': 'Admission grade', 'previous_grade': 'Previous qualification (grade)'}
Rows: 4,424 | columns: 37 | missing cells: 0 | exact duplicate rows: 0


,missing values
Target,0
Tuition fees up to date,0
Scholarship holder,0
Age at enrollment,0
Admission grade,0
Previous qualification (grade),0


,count
Target,
Graduate,2209
Dropout,1421
Enrolled,794


In [4]:
target_counts = data[COL['target']].value_counts(dropna=False)
display(target_counts.rename_axis('Target').to_frame('n'))
continuous = [COL[k] for k in ['age','admission_grade','previous_grade']]
display(data.groupby(COL['target'], dropna=False)[continuous].agg(['count','mean','std','median','min','max']).round(3))
for label in ['tuition','scholarship']:
    print(f'\n{COL[label]} by Target (counts):')
    display(pd.crosstab(data[COL[label]], data[COL['target']], margins=True, dropna=False))

,n
Target,
Graduate,2209
Dropout,1421
Enrolled,794


Age at enrollment                               Admission grade  \
                     count    mean    std median min max           count   
Target                                                                     
Dropout               1421  26.069  8.704   23.0  18  70            1421   
Enrolled               794  22.369  6.303   20.0  17  54             794   
Graduate              2209  21.784  6.693   19.0  17  62            2209   

                                              Previous qualification (grade)  \
             mean     std median   min    max                          count   
Target                                                                         
Dropout   124.961  15.125  123.6  95.0  190.0                           1421   
Enrolled  125.534  13.793  124.1  95.0  190.0                            794   
Graduate  128.794  14.071  127.4  95.0  190.0                           2209   

                                               
             mean     std median   min    max  
Target                                         
Dropout   131.114  12.873  133.1  95.0  190.0  
Enrolled  131.208  12.872  130.0  96.0  190.0  
Graduate  134.083  13.342  133.1  97.0  184.4


Tuition fees up to date by Target (counts):


Target,Dropout,Enrolled,Graduate,All
Tuition fees up to date,,,,
0,457,42,29,528
1,964,752,2180,3896
All,1421,794,2209,4424



Scholarship holder by Target (counts):


Target,Dropout,Enrolled,Graduate,All
Scholarship holder,,,,
0,1287,664,1374,3325
1,134,130,835,1099
All,1421,794,2209,4424


In [6]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

results = []
def record(name, test, statistic, p, effect, effect_name, n, detail='', ci=None):
    results.append({'analysis':name,'test':test,'statistic':statistic,'p_value':p,'effect_size':effect,'effect_size_metric':effect_name,'n':n,'confidence_interval_95':ci,'notes':detail})

def interpret(p):
    return 'Statistically significant at α=.05' if p < ALPHA else 'Not statistically significant at α=.05'

def chi_square_test(variable, label):
    subset = data[[variable, COL['target']]].dropna()
    table = pd.crosstab(subset[variable], subset[COL['target']])
    if table.shape[0] < 2 or table.shape[1] < 2:
        print(f'{label}: skipped; fewer than two observed categories in a variable.')
        return
    chi2, p, dof, expected = stats.chi2_contingency(table, correction=False)
    n = int(table.to_numpy().sum()); denom = min(table.shape[0]-1, table.shape[1]-1)
    v = float(np.sqrt(chi2/(n*denom))) if n and denom else np.nan
    sparse = int((expected < 5).sum())
    note = f'{sparse}/{expected.size} expected cells <5; minimum expected={expected.min():.2f}. '
    note += 'Use caution: chi-square approximation may be unreliable.' if sparse else 'Expected-count guideline met.'
    record(label, 'Pearson chi-square independence', float(chi2), float(p), v, "Cramér's V", n, note)
    print(f'{label}: χ²({dof})={chi2:.3f}, p={p:.4g}; Cramér V={v:.3f}; {interpret(p)}')
    print(note); display(table)

chi_square_test(COL['tuition'], 'H1: Tuition status × Target')
chi_square_test(COL['scholarship'], 'H2: Scholarship status × Target')

# H3: normality (Shapiro-Wilk, capped random sample) and Levene variance check.
groups = [g[COL['age']].dropna().to_numpy() for _,g in data[[COL['age'],COL['target']]].dropna().groupby(COL['target'])]
names = list(data[[COL['age'],COL['target']]].dropna().groupby(COL['target']).groups.keys())
if len(groups) >= 2 and all(len(g) >= 2 for g in groups):
    normality = []
    for name,g in zip(names,groups):
        sample = g if len(g) <= 5000 else np.random.default_rng(0).choice(g,5000,replace=False)
        normality.append((str(name), float(stats.shapiro(sample).pvalue) if len(sample)>=3 else np.nan))
    lev_stat, lev_p = stats.levene(*groups, center='median')
    normal_ok = all(np.isnan(p) or p >= ALPHA for _,p in normality)
    variance_ok = lev_p >= ALPHA
    print('Shapiro-Wilk p-values by Target:', normality)
    print(f'Levene test: statistic={lev_stat:.3f}, p={lev_p:.4g}')
    if normal_ok and variance_ok:
        stat,p = stats.f_oneway(*groups); grand = np.concatenate(groups).mean()
        ss_between = sum(len(g)*(g.mean()-grand)**2 for g in groups); ss_total = sum(((g-grand)**2).sum() for g in groups)
        eta2 = float(ss_between/ss_total) if ss_total else np.nan
        detail = 'ANOVA assumptions were not rejected by Shapiro-Wilk and Levene checks; tests have limited power and do not prove assumptions.'
        record('H3: Age at enrollment × Target','One-way ANOVA',float(stat),float(p),eta2,'eta-squared',sum(map(len,groups)),detail)
        print(f'One-way ANOVA: F={stat:.3f}, p={p:.4g}; η²={eta2:.3f}; {interpret(p)}')
    else:
        stat,p = stats.kruskal(*groups); n=sum(map(len,groups)); k=len(groups)
        eps = float(max(0,(stat-k+1)/(n-k))) if n>k else np.nan
        detail = 'Kruskal–Wallis selected because normality/variance checks raised concern; independent groups assumed. Epsilon-squared is an approximate effect size.'
        record('H3: Age at enrollment × Target','Kruskal-Wallis',float(stat),float(p),eps,'epsilon-squared',n,detail)
        print(f'Kruskal-Wallis: H={stat:.3f}, p={p:.4g}; ε²={eps:.3f}; {interpret(p)}')
        print('Normality/variance checks raised concern; inspect group distributions before interpretation.')
else:
    print('H3 skipped: insufficient nonmissing outcome groups.')

# H4: correlation; bootstrap percentile CI for Spearman.
pairs = data[[COL['admission_grade'],COL['previous_grade']]].dropna()
x,y = pairs.iloc[:,0].to_numpy(),pairs.iloc[:,1].to_numpy(); n=len(pairs)
if n >= 4 and np.std(x)>0 and np.std(y)>0:
    corr,p = stats.spearmanr(x,y)
    rng=np.random.default_rng(42); boots=[]
    for _ in range(2000):
        ix=rng.integers(0,n,n); bx,by=x[ix],y[ix]
        if np.std(bx)>0 and np.std(by)>0: boots.append(stats.spearmanr(bx,by).statistic)
    ci=[float(np.quantile(boots,.025)),float(np.quantile(boots,.975))] if boots else None
    record('H4: Admission grade ↔ previous qualification grade','Spearman rank correlation',float(corr),float(p),float(corr),'Spearman rho',n,'Bootstrap percentile confidence interval; association is not causation.',ci)
    print(f'Spearman ρ={corr:.3f}, p={p:.4g}, 95% bootstrap CI={ci}; {interpret(p)}')
else:
    print('H4 skipped: fewer than four complete pairs or no variation.')

if results:
    adjusted = multipletests([r['p_value'] for r in results], alpha=ALPHA, method='holm')[1]
    for row,p_adj in zip(results,adjusted):
        row['p_value_holm'] = float(p_adj)
        row['interpretation_raw'] = interpret(row['p_value'])
        row['interpretation_holm'] = interpret(p_adj)
display(pd.DataFrame(results))

H1: Tuition status × Target: χ²(2)=823.553, p=1.472e-179; Cramér V=0.431; Statistically significant at α=.05
0/6 expected cells <5; minimum expected=94.76. Expected-count guideline met.


Target,Dropout,Enrolled,Graduate
Tuition fees up to date,,,
0,457,42,29
1,964,752,2180


H2: Scholarship status × Target: χ²(2)=409.943, p=9.594e-90; Cramér V=0.304; Statistically significant at α=.05
0/6 expected cells <5; minimum expected=197.24. Expected-count guideline met.


Target,Dropout,Enrolled,Graduate
Scholarship holder,,,
0,1287,664,1374
1,134,130,835


Shapiro-Wilk p-values by Target: [('Dropout', 1.5872686630176095e-35), ('Enrolled', 5.718446970036319e-36), ('Graduate', 5.207900178756055e-58)]
Levene test: statistic=117.322, p=2.258e-50
Kruskal-Wallis: H=375.107, p=3.52e-82; ε²=0.084; Statistically significant at α=.05
Normality/variance checks raised concern; inspect group distributions before interpretation.
Spearman ρ=0.585, p=0, 95% bootstrap CI=[0.5605048009831561, 0.6082620967625182]; Statistically significant at α=.05


,analysis,test,statistic,p_value,effect_size,effect_size_metric,n,confidence_interval_95,notes,p_value_holm,interpretation_raw,interpretation_holm
0,H1: Tuition status × Target,Pearson chi-square independence,823.552724,1.471628e-179,0.431458,Cramér's V,4424,None,0/6 expected cells <5; minimum expected=94.76....,4.414885e-179,Statistically significant at α=.05,Statistically significant at α=.05
1,H2: Scholarship status × Target,Pearson chi-square independence,409.943055,9.593930e-90,0.304407,Cramér's V,4424,None,0/6 expected cells <5; minimum expected=197.24...,1.918786e-89,Statistically significant at α=.05,Statistically significant at α=.05
2,H3: Age at enrollment × Target,Kruskal-Wallis,375.106923,3.520194e-82,0.084394,epsilon-squared,4424,None,Kruskal–Wallis selected because normality/vari...,3.520194e-82,Statistically significant at α=.05,Statistically significant at α=.05
3,H4: Admission grade ↔ previous qualification g...,Spearman rank correlation,0.584624,0.000000e+00,0.584624,Spearman rho,4424,"[0.5605048009831561, 0.6082620967625182]",Bootstrap percentile confidence interval; asso...,0.000000e+00,Statistically significant at α=.05,Statistically significant at α=.05


In [7]:
try:
    import statsmodels.api as sm
    from sklearn.preprocessing import StandardScaler
    model_df = data[[COL['target'],COL['tuition'],COL['scholarship'],COL['age'],COL['admission_grade'],COL['previous_grade']]].dropna().copy()
    model_df = model_df[model_df[COL['target']].isin(['Graduate','Dropout'])]
    y_bin = (model_df[COL['target']] == 'Graduate').astype(int)
    X = model_df[[COL[k] for k in ['tuition','scholarship','age','admission_grade','previous_grade']]].astype(float)
    scaler = StandardScaler(); X_scaled = pd.DataFrame(scaler.fit_transform(X),columns=X.columns,index=X.index)
    X_design = sm.add_constant(X_scaled,has_constant='add')
    if y_bin.nunique() < 2: raise ValueError('Only one binary outcome remains after filtering.')
    fit = sm.Logit(y_bin,X_design).fit(disp=False, maxiter=200)
    ci = fit.conf_int(); reg = pd.DataFrame({'coefficient':fit.params,'odds_ratio':np.exp(fit.params),'OR_CI_low':np.exp(ci[0]),'OR_CI_high':np.exp(ci[1]),'p_value':fit.pvalues})
    print(f'Complete cases: {len(model_df)} | Graduate prevalence: {y_bin.mean():.3f} | pseudo-R²: {fit.prsquared:.3f}')
    display(reg.round(4))
except Exception as exc:
    print(f'Optional logistic regression not available for this data/model: {type(exc).__name__}: {exc}')

Complete cases: 3630 | Graduate prevalence: 0.609 | pseudo-R²: 0.242


,coefficient,odds_ratio,OR_CI_low,OR_CI_high,p_value
const,0.4300,1.5373,1.4055,1.6815,0.0000
Tuition fees up to date,1.1362,3.1150,2.7248,3.5611,0.0000
Scholarship holder,0.6589,1.9328,1.7549,2.1286,0.0000
Age at enrollment,-0.3865,0.6794,0.6264,0.7369,0.0000
Admission grade,0.2764,1.3184,1.1970,1.4520,0.0000
Previous qualification (grade),-0.0297,0.9708,0.8810,1.0697,0.5488


In [8]:
out_dir = Path.cwd()
results_df = pd.DataFrame(results)
csv_out = out_dir / 'week4_test_results.csv'
txt_out = out_dir / 'week4_test_results.txt'
results_df.to_csv(csv_out,index=False)
lines = [f'Week 4 Statistical Analysis Results (alpha={ALPHA})',f'Dataset: {csv_path.resolve()}',f'Rows: {len(data)} | columns: {data.shape[1]}',f'Missing cells: {int(data.isna().sum().sum())}', '']
for r in results:
    lines += [r['analysis'],f"Test: {r['test']} | statistic: {r['statistic']:.6g} | p: {r['p_value']:.6g} | Holm p: {r.get('p_value_holm',float('nan')):.6g}",f"Effect: {r['effect_size_metric']}={r['effect_size']:.6g} | n={r['n']} | {r['interpretation_holm']}",f"Notes: {r['notes']}",f"95% CI: {r['confidence_interval_95']}",'']
txt_out.write_text('\n'.join(lines),encoding='utf-8')
print(f'Wrote: {csv_out.resolve()}')
print(f'Wrote: {txt_out.resolve()}')
display(results_df)

Wrote: D:\Data_Analysis\week4_test_results.csv
Wrote: D:\Data_Analysis\week4_test_results.txt


,analysis,test,statistic,p_value,effect_size,effect_size_metric,n,confidence_interval_95,notes,p_value_holm,interpretation_raw,interpretation_holm
0,H1: Tuition status × Target,Pearson chi-square independence,823.552724,1.471628e-179,0.431458,Cramér's V,4424,None,0/6 expected cells <5; minimum expected=94.76....,4.414885e-179,Statistically significant at α=.05,Statistically significant at α=.05
1,H2: Scholarship status × Target,Pearson chi-square independence,409.943055,9.593930e-90,0.304407,Cramér's V,4424,None,0/6 expected cells <5; minimum expected=197.24...,1.918786e-89,Statistically significant at α=.05,Statistically significant at α=.05
2,H3: Age at enrollment × Target,Kruskal-Wallis,375.106923,3.520194e-82,0.084394,epsilon-squared,4424,None,Kruskal–Wallis selected because normality/vari...,3.520194e-82,Statistically significant at α=.05,Statistically significant at α=.05
3,H4: Admission grade ↔ previous qualification g...,Spearman rank correlation,0.584624,0.000000e+00,0.584624,Spearman rho,4424,"[0.5605048009831561, 0.6082620967625182]",Bootstrap percentile confidence interval; asso...,0.000000e+00,Statistically significant at α=.05,Statistically significant at α=.05
